# Kaggle House Prices | Regression Lab

**Goal:** benchmark strong models using leakage-safe 5-fold out-of-fold evaluation in log-price space, test a blended predictor, and create a valid `submission.csv`.

Competition: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

**Methodology:** all imputation/encoding is learned inside each fold; `Id` is excluded. The training target is `log1p(SalePrice)`; holdout RMSE in that space approximates the competition's log-price scoring closely. Do not tune on the public leaderboard. Run cells top to bottom.


In [ ]:
%pip -q install kagglehub lightgbm xgboost catboost

## 1. Load authorized competition data

First accept the competition rules in Kaggle. This cell attempts KaggleHub and falls back to Colab file upload (`train.csv`, `test.csv`, `sample_submission.csv`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import kagglehub

SEED = 42

# Authenticate Kaggle
kagglehub.login()

# Download competition dataset
data_dir = Path(
    kagglehub.competition_download(
        "house-prices-advanced-regression-techniques"
    )
)

# Verify files
required = ["train.csv", "test.csv", "sample_submission.csv"]

assert all(
    (data_dir / f).exists() for f in required
), f"Missing files in {data_dir}"

# Load datasets
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample = pd.read_csv(data_dir / "sample_submission.csv")

# Validate
assert {"Id", "SalePrice"}.issubset(train.columns)
assert "Id" in test.columns
assert "SalePrice" not in test.columns
assert len(sample) == len(test)

# Results
print("✅ Dataset loaded successfully!")
print("📂 Directory:", data_dir)
print("Train:", train.shape)
print("Test:", test.shape)
print("Submission:", sample.shape)

display(train.head(3))

## 2. Inspect data and construct safe features

In [ ]:
print('Top missing training features:')
display(train.isna().mean().sort_values(ascending=False).head(15).to_frame('missing_fraction'))
fig,ax=plt.subplots(figsize=(7,3)); ax.hist(np.log1p(train['SalePrice']),bins=30); ax.set_title('Log1p SalePrice'); plt.show()

def features(df):
    x=df.drop(columns=['Id','SalePrice'],errors='ignore').copy()
    # Codes represent classes, not continuous numeric measurements.
    for c in ['MSSubClass','MoSold']:
        if c in x: x[c]=x[c].astype('string').fillna('Missing').astype(str)
    # Features require no learned statistics and are computed per individual property.
    def col(name): return pd.to_numeric(x[name],errors='coerce').fillna(0) if name in x else pd.Series(0,index=x.index)
    x['TotalFinishedSF']=col('GrLivArea')+col('TotalBsmtSF')
    x['TotalBathrooms']=col('FullBath')+0.5*col('HalfBath')+col('BsmtFullBath')+0.5*col('BsmtHalfBath')
    x['TotalPorchSF']=sum((col(c) for c in ['OpenPorchSF','EnclosedPorch','3SsnPorch','ScreenPorch','WoodDeckSF']),start=pd.Series(0,index=x.index))
    x['HouseAgeAtSale']=(col('YrSold')-col('YearBuilt')).clip(lower=0)
    x['YearsSinceRemodel']=(col('YrSold')-col('YearRemodAdd')).clip(lower=0)
    return x
X=features(train); X_test=features(test); y=np.log1p(train['SalePrice'].to_numpy())
assert list(X.columns)==list(X_test.columns)
cat_cols=X.select_dtypes(include=['object','category','string']).columns.tolist()
num_cols=[c for c in X.columns if c not in cat_cols]
print(len(num_cols),'numeric;',len(cat_cols),'categorical features')

## 3. Fold-safe preprocessing and model definitions

In [ ]:
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import Ridge,ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

preprocessor=ColumnTransformer([
    ('num',Pipeline([('fill',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),num_cols),
    ('cat',Pipeline([('fill',SimpleImputer(strategy='constant',fill_value='Missing')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cols)
])
models={
 'Ridge':Ridge(alpha=15.0),
 'ElasticNet':ElasticNet(alpha=0.0007,l1_ratio=0.6,max_iter=30000,random_state=SEED),
 'RandomForest':RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features=0.85,n_jobs=-1,random_state=SEED),
 'LightGBM':LGBMRegressor(n_estimators=900,learning_rate=0.025,num_leaves=12,max_depth=-1,min_child_samples=12,colsample_bytree=0.85,subsample=0.8,subsample_freq=1,reg_lambda=3,verbose=-1,n_jobs=-1,random_state=SEED),
 'XGBoost':XGBRegressor(n_estimators=1100,max_depth=2,learning_rate=0.025,subsample=0.85,colsample_bytree=0.8,min_child_weight=3,reg_lambda=5,objective='reg:squarederror',tree_method='hist',n_jobs=-1,random_state=SEED),
 'CatBoost':CatBoostRegressor(iterations=1300,depth=5,learning_rate=0.03,l2_leaf_reg=5,loss_function='RMSE',verbose=False,thread_count=2,random_seed=SEED)
}
folds=list(KFold(n_splits=5,shuffle=True,random_state=SEED).split(X))
print('Models:',', '.join(models))

## 4. 5-fold OOF comparison

These are initial configurations, not optimized hyperparameters. CV scores are measured entirely out-of-fold. Models with unstable scores should not be trusted for blending.

In [ ]:
oof={}; fold_scores={}; test_preds={}
for name,estimator in models.items():
    pred=np.zeros(len(X)); test_fold_preds=[]; scores=[]
    for fold,(tr,va) in enumerate(folds,1):
        model=Pipeline([('prep',clone(preprocessor)),('model',clone(estimator))])
        model.fit(X.iloc[tr],y[tr])
        pred[va]=model.predict(X.iloc[va])
        test_fold_preds.append(model.predict(X_test))
        score=np.sqrt(mean_squared_error(y[va],pred[va])); scores.append(score)
    oof[name]=pred; fold_scores[name]=scores
    test_preds[name]=np.mean(test_fold_preds,axis=0)
    print(f'{name:13s} OOF={np.sqrt(mean_squared_error(y,pred)):.5f} | fold mean={np.mean(scores):.5f} ± {np.std(scores):.5f}')
results=pd.DataFrame([{'model':name,'oof_log_rmse':np.sqrt(mean_squared_error(y,oof[name])),'fold_std':np.std(fold_scores[name])} for name in models]).sort_values('oof_log_rmse')
display(results)
results.to_csv('/content/model_comparison.csv',index=False)

## 5. Compare ensemble against best individual model

A simple unoptimized blend of the top three is tested first. Its weights are not learned from the validation labels; this limits overfitting compared with freely optimizing blend weights on the same OOF labels. Prefer the single model unless blend OOF meaningfully improves.

In [ ]:
leaders = results['model'].head(3).tolist()

blend_oof = np.mean([oof[name] for name in leaders], axis=0)
blend_test = np.mean([test_preds[name] for name in leaders], axis=0)

blend_score = np.sqrt(mean_squared_error(y, blend_oof))

best_name = leaders[0]
best_score = float(results.iloc[0]['oof_log_rmse'])

print("Top three:", leaders)
print("Equal blend:", round(blend_score, 6))
print("Best single:", best_name, round(best_score, 6))

# Select best-performing approach
if blend_score < best_score:
    final_log_pred = blend_test
    chosen = 'equal_blend_' + '_'.join(leaders)
else:
    final_log_pred = test_preds[best_name]
    chosen = best_name

print('Selected:', chosen)

# Plot out-of-fold residuals
residuals = y - (
    blend_oof if chosen.startswith('equal_blend')
    else oof[best_name]
)

fig, ax = plt.subplots(figsize=(7, 3))

ax.scatter(y, residuals, alpha=0.4, s=14)
ax.axhline(0, linestyle='--')

ax.set(
    xlabel='Actual log1p(price)',
    ylabel='OOF residual',
    title='Out-of-fold residuals'
)

plt.tight_layout()
plt.show()

## 6. Generate Kaggle submission

Predictions are inverted from `log1p` with `expm1` and clipped positive. Kaggle leaderboard is an external check, not a replacement for cross-validation.

In [ ]:
final_prices=np.maximum(np.expm1(final_log_pred),1.0)
submission=sample[['Id']].copy()
submission['SalePrice']=final_prices
assert submission['Id'].equals(test['Id'])
assert submission['SalePrice'].notna().all() and np.isfinite(submission['SalePrice']).all()
assert (submission['SalePrice']>0).all()
submission.to_csv('/content/submission.csv',index=False)
print('Wrote /content/submission.csv using',chosen)
display(submission.head())
from google.colab import files
files.download('/content/submission.csv')

## Next experiments

1. Verify feature data types and domain-specific missingness handling; change one family of features at a time.
2. Run Optuna tuning **using internal folds only**; avoid selecting on the public Kaggle score.
3. Compare CatBoost native categoricals against one-hot version; CatBoost currently uses one-hot pipeline for consistency.
4. Compare robust regression and outlier treatment using explicit CV; do not remove anomalies only because the score rises on one split.
5. Evaluate blends via repeated CV or nested holdouts before tuning mixture weights.
6. Record experiments and seeds in a README; keep genuine CV vs leaderboard results clearly separate.

**Note:** A 5-fold test average is used for inference, rather than a separate full-data refit. This aligns with fold ensemble predictions and is a valid submission strategy.


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

# Select top 4 models
leaders = results["model"].head(4).tolist()

P = np.column_stack([oof[name] for name in leaders])
P_test = np.column_stack([test_preds[name] for name in leaders])

def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

# Baselines
single_score = rmse(y, P[:, 0])
equal_score = rmse(y, P.mean(axis=1))

# Learn blend weights using cross-fitting
meta_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

blend_oof = np.zeros(len(y))
weight_history = []

for tr, va in meta_cv.split(P):

    def objective(weights):
        prediction = P[tr] @ weights

        return np.mean((y[tr] - prediction) ** 2)

    n = len(leaders)

    result = minimize(
        objective,
        x0=np.ones(n) / n,
        method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints=[{
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1
        }],
        options={"maxiter": 1000}
    )

    weights = result.x if result.success else np.ones(n) / n

    blend_oof[va] = P[va] @ weights
    weight_history.append(weights)

crossfit_score = rmse(y, blend_oof)

# Fit final weights using all training OOF predictions
n = len(leaders)

final_result = minimize(
    lambda w: np.mean((y - P @ w) ** 2),
    x0=np.ones(n) / n,
    method="SLSQP",
    bounds=[(0, 1)] * n,
    constraints=[{
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    }],
    options={"maxiter": 1000}
)

final_weights = (
    final_result.x
    if final_result.success
    else np.ones(n) / n
)

print("\nMODEL COMPARISON")
print("-" * 45)

print(f"Best single:       {single_score:.6f}")
print(f"Equal blend (4):   {equal_score:.6f}")
print(f"Cross-fit blend:   {crossfit_score:.6f}")

print("\nOPTIMIZED MODEL WEIGHTS")
print("-" * 45)

for name, weight in zip(leaders, final_weights):
    print(f"{name:15s}: {weight:.2%}")

# Compare existing best approach with experimental blend
previous_score = rmse(y, oof[results.iloc[0]["model"]])

if crossfit_score < previous_score:
    final_log_pred = P_test @ final_weights
    chosen = "optimized_weighted_ensemble"
    print("\n✅ Experimental blend selected")
else:
    chosen = results.iloc[0]["model"]
    final_log_pred = test_preds[chosen]
    print("\n✅ Best original single model retained")

# Generate submission
submission = sample[["Id"]].copy()

submission["SalePrice"] = np.maximum(
    np.expm1(final_log_pred),
    1.0
)

submission.to_csv(
    "/content/submission_optimized.csv",
    index=False
)

print("\n✅ submission_optimized.csv generated")

display(submission.head())

In [ ]:
import optuna
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

cv = KFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 1200, step=100),
        "depth": trial.suggest_int("depth", 4, 6),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.03, 0.12, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1, 10, log=True
        ),
        "loss_function": "RMSE",
        "random_seed": 42,
        "thread_count": 2,
        "verbose": False,
        "allow_writing_files": False
    }

    predictions = np.zeros(len(y))

    for tr, va in cv.split(X):
        model = CatBoostRegressor(**params)

        model.fit(
            X.iloc[tr],
            y[tr],
            cat_features=cat_cols,
            eval_set=(X.iloc[va], y[va]),
            early_stopping_rounds=75,
            verbose=False
        )

        predictions[va] = model.predict(X.iloc[va])

    return np.sqrt(mean_squared_error(y, predictions))


study_fast = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_fast.optimize(
    objective,
    n_trials=12,
    show_progress_bar=True
)

print("\nFAST OPTUNA RESULTS")
print("-" * 40)
print(f"Best 3-fold CV: {study_fast.best_value:.6f}")
print("Best parameters:", study_fast.best_params)

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

best_params = study_fast.best_params.copy()

cv_5 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

optimized_oof = np.zeros(len(y))
optimized_test_folds = []

for fold, (tr, va) in enumerate(cv_5.split(X), 1):

    model = CatBoostRegressor(
        **best_params,
        loss_function="RMSE",
        random_seed=42,
        thread_count=2,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X.iloc[tr],
        y[tr],
        cat_features=cat_cols,
        eval_set=(X.iloc[va], y[va]),
        early_stopping_rounds=75,
        verbose=False
    )

    optimized_oof[va] = model.predict(X.iloc[va])
    optimized_test_folds.append(
        model.predict(X_test)
    )

    fold_rmse = np.sqrt(
        mean_squared_error(y[va], optimized_oof[va])
    )

    print(f"Fold {fold}: {fold_rmse:.6f}")

optimized_score = np.sqrt(
    mean_squared_error(y, optimized_oof)
)

optimized_test = np.mean(
    optimized_test_folds,
    axis=0
)

print("\n" + "=" * 45)
print("5-FOLD CATBOOST RESULTS")
print("=" * 45)

print(f"Original CatBoost: 0.123416")
print(f"Optimized CatBoost: {optimized_score:.6f}")

if optimized_score < 0.123416:
    print("✅ Optimized CatBoost has a lower recorded CV score!")
else:
    print("❌ Original CatBoost remains the benchmark")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Analyze optimized CatBoost's OOF predictions
error_analysis = train[
    ["Id", "SalePrice", "OverallQual", "GrLivArea",
     "Neighborhood", "YearBuilt"]
].copy()

error_analysis["PredictedPrice"] = np.expm1(optimized_oof)

error_analysis["LogError"] = (
    np.log1p(error_analysis["SalePrice"])
    - optimized_oof
)

error_analysis["AbsoluteLogError"] = (
    error_analysis["LogError"].abs()
)

error_analysis["PercentageError"] = (
    100
    * (error_analysis["PredictedPrice"] - error_analysis["SalePrice"])
    / error_analysis["SalePrice"]
)

# Largest prediction errors
worst_predictions = error_analysis.nlargest(
    20, "AbsoluteLogError"
)

print("TOP 20 WORST PREDICTIONS")
display(worst_predictions)

# Error by neighborhood
neighborhood_errors = (
    error_analysis.groupby("Neighborhood")
    .agg(
        Count=("Id", "count"),
        MeanAbsoluteLogError=("AbsoluteLogError", "mean")
    )
    .sort_values(
        "MeanAbsoluteLogError",
        ascending=False
    )
)

print("\nNEIGHBORHOOD ERROR ANALYSIS")
display(neighborhood_errors.head(15))

# Plot residuals
plt.figure(figsize=(9, 5))

plt.scatter(
    error_analysis["SalePrice"],
    error_analysis["LogError"],
    alpha=0.4
)

plt.axhline(0, linestyle="--")

plt.xlabel("Actual Sale Price")
plt.ylabel("Prediction Error (Log Scale)")
plt.title("CatBoost: Out-of-Fold Prediction Errors")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Original and optimized CatBoost predictions
original_oof = np.asarray(oof["CatBoost"]).ravel()
tuned_oof = np.asarray(optimized_oof).ravel()
target = np.asarray(y).ravel()

assert len(original_oof) == len(tuned_oof) == len(target) == len(train)

comparison = train[
    ["Id", "SalePrice", "OverallQual",
     "GrLivArea", "Neighborhood", "YearBuilt"]
].copy()

comparison["OriginalPrediction"] = np.expm1(original_oof)
comparison["TunedPrediction"] = np.expm1(tuned_oof)

comparison["OriginalError"] = np.abs(target - original_oof)
comparison["TunedError"] = np.abs(target - tuned_oof)

comparison["OriginalSquaredError"] = (
    target - original_oof
) ** 2

comparison["TunedSquaredError"] = (
    target - tuned_oof
) ** 2

# Overall performance
original_rmse = np.sqrt(
    comparison["OriginalSquaredError"].mean()
)

tuned_rmse = np.sqrt(
    comparison["TunedSquaredError"].mean()
)

print("=" * 55)
print("ORIGINAL VS OPTIMIZED CATBOOST")
print("=" * 55)

print(f"Original CatBoost:  {original_rmse:.6f}")
print(f"Optimized CatBoost: {tuned_rmse:.6f}")

# Compare the most problematic houses
problem_ids = [524, 1299, 31, 633, 496, 1183]

print("\nPROBLEMATIC PROPERTY COMPARISON")

display(
    comparison[
        comparison["Id"].isin(problem_ids)
    ].sort_values("OriginalError", ascending=False)
)

# Original model's 20 largest errors
print("\nORIGINAL CATBOOST — WORST 20 PREDICTIONS")

display(
    comparison.nlargest(20, "OriginalError")[
        [
            "Id",
            "SalePrice",
            "OriginalPrediction",
            "OriginalError",
            "OverallQual",
            "GrLivArea",
            "Neighborhood"
        ]
    ]
)

# Quantify impact of two extreme observations
extreme_mask = comparison["Id"].isin([524, 1299])

total_squared_error = comparison[
    "OriginalSquaredError"
].sum()

extreme_squared_error = comparison.loc[
    extreme_mask, "OriginalSquaredError"
].sum()

contribution = (
    extreme_squared_error / total_squared_error * 100
)

print("\nOUTLIER CONTRIBUTION")

print(
    f"IDs 524 and 1299 account for "
    f"{contribution:.2f}% of original model squared error"
)

# Diagnostic only: excluded from evaluation, NOT training
remaining = comparison.loc[~extreme_mask]

diagnostic_rmse = np.sqrt(
    remaining["OriginalSquaredError"].mean()
)

print(
    f"RMSE excluding these 2 evaluation rows: "
    f"{diagnostic_rmse:.6f}"
)

print(
    "\nNote: This reduced RMSE is diagnostic only, "
    "not a valid replacement for the full CV score."
)

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

SEED = 42

# Existing prepared data:
# X, X_test, y, cat_cols

problem_ids = {524, 1299}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Use fixed parameters for a controlled comparison
params = {
    "iterations": 900,
    "depth": 4,
    "learning_rate": 0.0696465652082127,
    "l2_leaf_reg": 1.4808945119975185,
    "loss_function": "RMSE",
    "random_seed": SEED,
    "thread_count": 2,
    "verbose": False,
    "allow_writing_files": False
}

oof_control = np.zeros(len(X))
oof_filtered = np.zeros(len(X))

train_ids = train["Id"].to_numpy()

for fold, (tr, va) in enumerate(cv.split(X), 1):

    # CONTROL: train with all training records
    model_control = CatBoostRegressor(**params)

    model_control.fit(
        X.iloc[tr],
        y[tr],
        cat_features=cat_cols,
        verbose=False
    )

    oof_control[va] = model_control.predict(X.iloc[va])

    # EXPERIMENT: exclude two houses from training only
    keep = ~np.isin(train_ids[tr], list(problem_ids))
    filtered_tr = tr[keep]

    model_filtered = CatBoostRegressor(**params)

    model_filtered.fit(
        X.iloc[filtered_tr],
        y[filtered_tr],
        cat_features=cat_cols,
        verbose=False
    )

    # IMPORTANT: evaluate on the SAME validation rows
    oof_filtered[va] = model_filtered.predict(X.iloc[va])

    control_fold = np.sqrt(
        mean_squared_error(y[va], oof_control[va])
    )

    filtered_fold = np.sqrt(
        mean_squared_error(y[va], oof_filtered[va])
    )

    print(
        f"Fold {fold}: "
        f"Control={control_fold:.6f} | "
        f"Filtered={filtered_fold:.6f}"
    )

control_score = np.sqrt(
    mean_squared_error(y, oof_control)
)

filtered_score = np.sqrt(
    mean_squared_error(y, oof_filtered)
)

print("\n" + "=" * 55)
print("OUTLIER EXPERIMENT RESULTS")
print("=" * 55)

print(f"Original historical benchmark: 0.123416")
print(f"Controlled baseline:           {control_score:.6f}")
print(f"Outlier-filtered model:        {filtered_score:.6f}")
print(f"Difference (control-filtered): {control_score-filtered_score:+.6f}")

if filtered_score < control_score:
    print("\n✅ Filtering improved this CV experiment.")
else:
    print("\n❌ Filtering did not improve this CV experiment.")

print(
    "\nNote: Both models were evaluated on all 1,460 "
    "observations, including IDs 524 and 1299."
)

In [ ]:
print("=" * 55)
print("ORIGINAL MODEL RESULTS")
print("=" * 55)

display(results)

print("\nAvailable OOF models:")
print(list(oof.keys()))

print("\nOriginal CatBoost score:")
print(
    results.loc[
        results["model"] == "CatBoost",
        ["model", "oof_log_rmse"]
    ]
)

print("\nCurrent feature matrix:")
print("Shape:", X.shape)
print("Categorical columns:", len(cat_cols))

print("\nFeature types:")
display(X.dtypes.value_counts())

print("\nOutlier fold locations:")

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (_, va) in enumerate(cv.split(X), 1):
    ids = train.iloc[va]["Id"]
    found = ids[ids.isin([524, 1299])].tolist()

    if found:
        print(f"Fold {fold}: {found}")

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

SEED = 42

# -------------------------------
# 1. Feature engineering
# -------------------------------

def add_house_features(df):
    df = df.copy()

    def numeric(name):
        return pd.to_numeric(
            df[name], errors="coerce"
        ).fillna(0)

    df["TotalSF"] = (
        numeric("TotalBsmtSF")
        + numeric("1stFlrSF")
        + numeric("2ndFlrSF")
    )

    df["TotalBathrooms"] = (
        numeric("FullBath")
        + 0.5 * numeric("HalfBath")
        + numeric("BsmtFullBath")
        + 0.5 * numeric("BsmtHalfBath")
    )

    df["HouseAge"] = (
        numeric("YrSold") - numeric("YearBuilt")
    ).clip(lower=0)

    df["YearsSinceRemodel"] = (
        numeric("YrSold") - numeric("YearRemodAdd")
    ).clip(lower=0)

    df["TotalPorchSF"] = (
        numeric("OpenPorchSF")
        + numeric("EnclosedPorch")
        + numeric("3SsnPorch")
        + numeric("ScreenPorch")
        + numeric("WoodDeckSF")
    )

    df["QualityArea"] = (
        numeric("OverallQual")
        * numeric("GrLivArea")
    )

    df["NeighborhoodQuality"] = (
        df["Neighborhood"].astype(str)
        + "_"
        + df["OverallQual"].astype(str)
    )

    return df


X_base = X.copy()
X_engineered = add_house_features(X)

base_cat_cols = X_base.select_dtypes(
    include=["object", "category"]
).columns.tolist()

eng_cat_cols = X_engineered.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Original features:", X_base.shape[1])
print("Engineered features:", X_engineered.shape[1])


# -------------------------------
# 2. Controlled validation
# -------------------------------

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

params = {
    "iterations": 900,
    "depth": 4,
    "learning_rate": 0.0696465652082127,
    "l2_leaf_reg": 1.4808945119975185,
    "loss_function": "RMSE",
    "random_seed": SEED,
    "thread_count": 2,
    "verbose": False,
    "allow_writing_files": False
}

baseline_oof = np.zeros(len(y))
engineered_oof = np.zeros(len(y))

for fold, (tr, va) in enumerate(cv.split(X_base), 1):

    baseline_model = CatBoostRegressor(**params)

    baseline_model.fit(
        X_base.iloc[tr],
        y[tr],
        cat_features=base_cat_cols,
        verbose=False
    )

    baseline_oof[va] = baseline_model.predict(
        X_base.iloc[va]
    )

    engineered_model = CatBoostRegressor(**params)

    engineered_model.fit(
        X_engineered.iloc[tr],
        y[tr],
        cat_features=eng_cat_cols,
        verbose=False
    )

    engineered_oof[va] = engineered_model.predict(
        X_engineered.iloc[va]
    )

    baseline_fold = np.sqrt(
        mean_squared_error(y[va], baseline_oof[va])
    )

    engineered_fold = np.sqrt(
        mean_squared_error(y[va], engineered_oof[va])
    )

    print(
        f"Fold {fold}: "
        f"Baseline={baseline_fold:.6f} | "
        f"Engineered={engineered_fold:.6f}"
    )


# -------------------------------
# 3. Final comparison
# -------------------------------

baseline_score = np.sqrt(
    mean_squared_error(y, baseline_oof)
)

engineered_score = np.sqrt(
    mean_squared_error(y, engineered_oof)
)

print("\n" + "=" * 55)
print("FEATURE ENGINEERING EXPERIMENT")
print("=" * 55)

print(f"Controlled baseline: {baseline_score:.6f}")
print(f"Engineered model:    {engineered_score:.6f}")

print(
    f"Improvement:         "
    f"{baseline_score - engineered_score:+.6f}"
)

if engineered_score < baseline_score:
    print("\n✅ Feature engineering improved CV performance.")
else:
    print("\n❌ Original feature set performed better.")